# Draft selection, tools, streaming, and handoffs: practice

Create three writers → stream or gather drafts → choose one → convert writer agents into tools → coordinate from a manager → hand off formatting/delivery.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [2_openai/2_lab2.ipynb](../../../../../2_openai/2_lab2.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Configure a real SDK agent and inspect its tool schema

Agent holds instructions/model/tools. The function_tool decorator constructs a schema from the signature and docstring. Constructing this configuration does not run the agent or contact a provider.

**Predict:** Which argument names will appear in the registered tool schema?

In [ ]:
from agents import Agent, function_tool, set_tracing_disabled
set_tracing_disabled(True)
@function_tool
def add_numbers(a: int, b: int) -> int:
    """Add two integers."""
    return a + b
agent = Agent(name="Practice teacher", instructions="Use add_numbers for addition.",
              model="gpt-4o-mini", tools=[add_numbers])
print(agent.name, agent.tools[0].name)
print(sorted(agent.tools[0].params_json_schema["properties"]))
assert sorted(agent.tools[0].params_json_schema["properties"]) == ["a", "b"]

**Check:** a and b. The schema advertises the function; Runner executes the model/tool workflow when deliberately invoked.

**Change one thing:** Change the docstring and inspect the tool description. Find Runner.run in the source: that is where the interaction loop actually begins.

## 2. Await vs. concurrent work

Calling an async function creates a coroutine. Await executes/waits for it. gather schedules independent coroutines and returns results in input order. Notebook cells support top-level await.

**Predict:** Will results follow completion order or input order?

In [ ]:
import asyncio
async def work(label, delay):
    await asyncio.sleep(delay)  # Yield control while waiting.
    return label
results = await asyncio.gather(work("A", 0.002), work("B", 0))
print(results)
assert results == ["A", "B"]

**Check:** Input order. In a standalone script, wrap work in async def main and call asyncio.run(main()); do not do that inside a running notebook loop.

**Change one thing:** Run the two functions with separate awaits. Explain why a dependent stage cannot start until it has its input.

## 3. Streaming sends partial progress

A generator yields multiple updates. A streamed API can include multiple event types, so a UI should select the text events rather than assume every event is text.

**Predict:** How many text chunks are selected?

In [ ]:
events = [
    {"type": "status", "value": "started"},
    {"type": "text", "value": "Hello "},
    {"type": "text", "value": "learner"},
]
def text_chunks(items):
    for event in items:
        if event["type"] == "text":
            yield event["value"]
chunks = list(text_chunks(events))
print(chunks, "→", "".join(chunks))
assert "".join(chunks) == "Hello learner"

**Check:** Two. yield preserves progress; return ends an ordinary function.

**Change one thing:** Insert a non-text event between the chunks. Confirm it does not become displayed answer text.

## 4. Compare alternatives and decode a ranking

Parallel candidates answer the same question. A separate selection step chooses among them. A sample JSON result lets you practice decoding without trusting a model judge.

**Predict:** Why subtract one from a competitor number?

In [ ]:
import json
competitors = ["writer_a", "writer_b", "writer_c"]
answers = ["Detailed answer", "Short answer", "Answer with an example"]
assert len(competitors) == len(answers)
for label, answer in zip(competitors, answers):
    print(label, "→", answer)
ranking = json.loads('{"results": ["3", "1", "2"]}')
ranks = [int(value) for value in ranking["results"]]
# Validate that every candidate appears exactly once.
assert sorted(ranks) == list(range(1, len(competitors) + 1))
ordered = [competitors[number - 1] for number in ranks]
print("Ranking:", ordered)
assert ordered[0] == "writer_c"

**Check:** The judge uses one-based labels; list indices start at zero. A ranking is a judgment, not an objective accuracy proof.

**Change one thing:** Reorder the sample ranking and update the expected winner. Try a duplicate rank and see the validation reject it.

## 5. Agent as a tool vs. handoff

A tool returns its result to the calling agent. A handoff transfers control to the receiving agent. This trace uses ordinary Python to make that distinction visible.

**Predict:** After the tool finishes, who produces the next step?

In [ ]:
def specialist_tool(topic):
    return f"Specialist result for {topic}"
tool_trace = ["manager starts", specialist_tool("Python"), "manager resumes"]
handoff_trace = ["manager starts", "manager hands off", "specialist controls remaining reply"]
print("Tool:", tool_trace)
print("Handoff:", handoff_trace)
assert tool_trace[-1] == "manager resumes"

**Check:** The manager resumes after a tool. The specialist takes control after a handoff.

**Change one thing:** Sketch which pattern fits a writer consulting a researcher, and which fits routing a customer to support.

## 6. A tool schema describes; a dispatcher executes

The model can request a function call. Your application decodes arguments, recognizes the name, executes the function, and returns a matching tool result. No model is needed to practice this boundary.

**Predict:** Does defining the schema call the function?

In [ ]:
import json
records = []
def record_interest(email):
    records.append(email)  # In memory only; does not send email or notifications.
    return {"recorded": True}
schema = {
    "type": "function", "function": {
        "name": "record_interest", "description": "Record a supplied address",
        "parameters": {"type": "object", "properties": {"email": {"type": "string"}},
                       "required": ["email"], "additionalProperties": False},
    },
}
assert records == []
requested = {"id": "call_1", "name": "record_interest", "arguments": '{"email":"learner@example.com"}'}
allowed = {"record_interest": record_interest}  # An explicit registry of permitted functions.
if requested["name"] not in allowed:
    raise ValueError("Unknown tool")
result = allowed[requested["name"]](**json.loads(requested["arguments"]))
tool_message = {"role": "tool", "tool_call_id": requested["id"], "content": json.dumps(result)}
print(tool_message)
assert records == ["learner@example.com"]

**Check:** No. Only the explicit dispatcher call changes records. Keep the assistant tool-call message before the matching tool result in a full conversation.

**Change one thing:** Change the requested name to an unknown function and observe the error. Then add a harmless second tool to the registry.

## 7. Drafting and delivering are separate steps

An agent that formats text is different from a tool that sends email or notifications. This exercise records a delivery intention in memory only.

**Predict:** Will constructing this draft contact anyone?

In [ ]:
outbox = []
draft = {"to": "learner@example.com", "subject": "Practice report", "body": "Fictional findings."}
assert outbox == []
def dry_run_send(message):
    outbox.append({"status": "dry_run", "subject": message["subject"]})
    return {"status": "dry_run"}
result = dry_run_send(draft)
print(result, outbox)
assert len(outbox) == 1

**Check:** No. Neither this draft nor this in-memory dry run sends anything. A provider send/post call is the delivery boundary.

**Change one thing:** Format the body as HTML but keep dry_run_send unchanged. Explain where real delivery would begin.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.